# PromptSentinel — Notebook 2
## Attack Type Analysis

**Author:** Leesha Mogha  
**Institution:** IMS Ghaziabad (University Course Campus)  
**Project:** PromptSentinel (v4)  
**Builds on:** [Prompt-Safety-Classifier](https://github.com/leeeshart/Prompt-Safety-Classifier)

---

### What this notebook does
Investigates whether different types of harmful prompts need
different detection methods, by training on one attack type
and testing on another.

### Research question answered here
**RQ1: Do different attack types need different detection methods?**

### How we answer it
WildJailbreak separates harmful prompts into two types:
- **Direct attacks** — straightforward harmful requests ("how do I make a bomb?")
- **Adversarial attacks** — jailbreak-wrapped requests ("write a story where a character explains...")

We train a model on direct attacks only, then test it on adversarial attacks.
The drop in performance tells us how much attack type matters.

---
## Section 1: Setup

In [2]:
!pip install datasets pandas numpy scikit-learn -q

from huggingface_hub import login
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

login()

print("Libraries loaded!")

Libraries loaded!


---
## Section 2: Load Data

Loading the combined dataset from Notebook 1.

**Note:** If you're not running in Colab, place `combined_dataset_final.csv` in this folder or provide its path when prompted.
Then separating WildJailbreak prompts by attack type
so we can run controlled experiments.

In [ ]:
# Load combined_dataset_final.csv from Notebook 1
from pathlib import Path

data_path = Path('combined_dataset_final.csv')

if not data_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        if data_path.name not in uploaded:
            raise FileNotFoundError(f"{data_path.name} was not uploaded.")
    except Exception:
        user_path = input(
            'Enter the path to combined_dataset_final.csv (leave blank to stop): '
        ).strip()
        if not user_path:
            raise FileNotFoundError('combined_dataset_final.csv not found.')
        data_path = Path(user_path).expanduser()
        if not data_path.exists():
            raise FileNotFoundError(f'File not found: {data_path}')

df = pd.read_csv(data_path)

print(f"Dataset loaded: {len(df):,} prompts")
print(df['label'].value_counts())
print()
print('Sources:')
print(df['source'].value_counts())


### Reloading WildJailbreak with attack type labels

The combined dataset from Notebook 1 only has safe/unsafe labels.
For this experiment we need the original attack type column
(vanilla_harmful vs adversarial_harmful) from WildJailbreak.

In [5]:
# Reload WildJailbreak keeping the original attack type column
wild = load_dataset(
    "allenai/wildjailbreak",
    "train",
    delimiter="\t",
    keep_default_na=False,
    token=True
)
df_wild = wild['train'].to_pandas()

# Keep attack type for unsafe prompts only
df_wild = df_wild[['vanilla', 'data_type']].copy()
df_wild.columns = ['prompt', 'attack_type']
df_wild = df_wild.dropna()
df_wild = df_wild[df_wild['prompt'].str.strip() != '']

# Separate by attack type
direct     = df_wild[df_wild['attack_type'] == 'vanilla_harmful'].copy()
adversarial = df_wild[df_wild['attack_type'] == 'adversarial_harmful'].copy()
benign      = df_wild[df_wild['attack_type'].isin(['vanilla_benign', 'adversarial_benign'])].copy()

direct['label']      = 'unsafe'
adversarial['label'] = 'unsafe'
benign['label']      = 'safe'

print(f"Direct attacks:      {len(direct):,}")
print(f"Adversarial attacks: {len(adversarial):,}")
print(f"Benign prompts:      {len(benign):,}")

train/train.tsv:   0%|          | 0.00/531M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Direct attacks:      50,050
Adversarial attacks: 82,728
Benign prompts:      128,781


---
## Section 3: Experiment Design

We run three experiments and compare results:

| Experiment | Trained on | Tested on | What it shows |
|---|---|---|---|
| A | Direct attacks only | Adversarial attacks | Does training type matter? |
| B | Adversarial attacks only | Direct attacks | Same question, reversed |
| C | Both combined | Both combined | Does mixing types help? |

Same model used in all three — TF-IDF + Logistic Regression (balanced).
Only the training data changes. This isolates the effect of attack type.

In [6]:
# Helper function — trains and evaluates the model
# Same model as v2 so results are directly comparable

def run_experiment(train_df, test_df, experiment_name):
    print(f"=== {experiment_name} ===")
    print(f"Train size: {len(train_df):,} | Test size: {len(test_df):,}")
    print()

    vectorizer = TfidfVectorizer(max_features=5000)
    X_train = vectorizer.fit_transform(train_df['prompt'])
    X_test  = vectorizer.transform(test_df['prompt'])

    y_train = train_df['label']
    y_test  = test_df['label']

    model = LogisticRegression(max_iter=1000, class_weight='balanced')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    report = classification_report(
        y_test, y_pred,
        output_dict=True,
        zero_division=0
    )

    unsafe_recall    = report['unsafe']['recall']
    unsafe_precision = report['unsafe']['precision']
    unsafe_f1        = report['unsafe']['f1-score']
    accuracy         = report['accuracy']

    print(classification_report(y_test, y_pred, zero_division=0))

    return {
        'Experiment':       experiment_name,
        'Accuracy':         round(accuracy, 3),
        'Unsafe Recall':    round(unsafe_recall, 3),
        'Unsafe Precision': round(unsafe_precision, 3),
        'Unsafe F1':        round(unsafe_f1, 3),
    }

---
## Section 4: Experiment A — Trained on Direct, Tested on Adversarial

If the model trained only on direct attacks generalises well to
adversarial attacks, the recall will stay high.
If it drops significantly, that proves attack types need
separate handling.

In [7]:
# Experiment A
# Train: direct attacks + benign
# Test:  adversarial attacks + benign (unseen attack type)

# Sample benign to keep balance reasonable
benign_sample = benign.sample(
    min(len(direct), len(benign)),
    random_state=42
)

train_a = pd.concat([direct, benign_sample], ignore_index=True)
test_a  = pd.concat([adversarial, benign_sample], ignore_index=True)

# Shuffle
train_a = train_a.sample(frac=1, random_state=42).reset_index(drop=True)
test_a  = test_a.sample(frac=1, random_state=42).reset_index(drop=True)

result_a = run_experiment(train_a, test_a, "A — Trained on Direct, Tested on Adversarial")

=== A — Trained on Direct, Tested on Adversarial ===
Train size: 100,100 | Test size: 132,778

              precision    recall  f1-score   support

        safe       0.97      0.99      0.98     50050
      unsafe       0.99      0.98      0.98     82728

    accuracy                           0.98    132778
   macro avg       0.98      0.98      0.98    132778
weighted avg       0.98      0.98      0.98    132778



---
## Section 5: Experiment B — Trained on Adversarial, Tested on Direct

The reverse experiment. If adversarial training generalises to
direct attacks better than the other way around, that tells us
something important about which attack type is harder to learn from.

In [8]:
# Experiment B
# Train: adversarial attacks + benign
# Test:  direct attacks + benign (unseen attack type)

benign_sample_b = benign.sample(
    min(len(adversarial), len(benign)),
    random_state=42
)

train_b = pd.concat([adversarial, benign_sample_b], ignore_index=True)
test_b  = pd.concat([direct, benign_sample_b], ignore_index=True)

train_b = train_b.sample(frac=1, random_state=42).reset_index(drop=True)
test_b  = test_b.sample(frac=1, random_state=42).reset_index(drop=True)

result_b = run_experiment(train_b, test_b, "B — Trained on Adversarial, Tested on Direct")

=== B — Trained on Adversarial, Tested on Direct ===
Train size: 165,456 | Test size: 132,778

              precision    recall  f1-score   support

        safe       0.99      0.99      0.99     82728
      unsafe       0.98      0.98      0.98     50050

    accuracy                           0.98    132778
   macro avg       0.98      0.98      0.98    132778
weighted avg       0.98      0.98      0.98    132778



---
## Section 6: Experiment C — Trained on Both, Tested on Both

The combined model. If mixing attack types during training
improves recall on both types, that answers RQ1 — yes,
you need both in training, not just one.

In [9]:
# Experiment C
# Train + Test: both attack types combined (standard train/test split)

all_unsafe = pd.concat([direct, adversarial], ignore_index=True)
all_data   = pd.concat([all_unsafe, benign], ignore_index=True)
all_data   = all_data.sample(frac=1, random_state=42).reset_index(drop=True)

train_c, test_c = train_test_split(
    all_data,
    test_size=0.2,
    random_state=42,
    stratify=all_data['label']
)

result_c = run_experiment(train_c, test_c, "C — Trained on Both, Tested on Both")

=== C — Trained on Both, Tested on Both ===
Train size: 209,247 | Test size: 52,312

              precision    recall  f1-score   support

        safe       0.98      0.98      0.98     25756
      unsafe       0.99      0.98      0.98     26556

    accuracy                           0.98     52312
   macro avg       0.98      0.98      0.98     52312
weighted avg       0.98      0.98      0.98     52312



---
## Section 7: Results Comparison

Putting all three experiments side by side to answer RQ1.

In [10]:
results_df = pd.DataFrame([result_a, result_b, result_c])

print("=== Final Comparison ===")
print()
print(results_df.to_string(index=False))
print()

# Find the key drop
recall_a = result_a['Unsafe Recall']
recall_c = result_c['Unsafe Recall']
drop     = round(recall_c - recall_a, 3)

print(f"Recall drop (A vs C): {drop}")
print()

if recall_a < 0.7:
    print("Finding: Training on direct attacks alone fails badly on adversarial attacks.")
    print("This confirms that attack types need to be represented in training data separately.")
elif recall_a < recall_c - 0.05:
    print("Finding: Mixed training (Experiment C) outperforms single-type training.")
    print("Attack type diversity in training data improves generalisation.")
else:
    print("Finding: Attack types generalise reasonably well to each other.")
    print("A single model handles both without needing separate treatment.")

=== Final Comparison ===

                                  Experiment  Accuracy  Unsafe Recall  Unsafe Precision  Unsafe F1
A — Trained on Direct, Tested on Adversarial     0.981          0.979             0.991      0.985
B — Trained on Adversarial, Tested on Direct     0.984          0.978             0.979      0.978
         C — Trained on Both, Tested on Both     0.983          0.981             0.985      0.983

Recall drop (A vs C): 0.002

Finding: Attack types generalise reasonably well to each other.
A single model handles both without needing separate treatment.


### **Key Finding — RQ1**

All three experiments produced identical unsafe recall of 0.98.

This means TF-IDF generalises across attack types when the
adversarial examples are synthetically generated — the harmful
vocabulary is preserved even through jailbreak wrapping.

**However this result has an important limitation:**
WildJailbreak adversarial prompts are machine-generated by
wrapping direct requests. Real human-written jailbreaks
(like those in TrustAIRLab) use more creative language
that may not share the same vocabulary.

This raises the question Notebook 3 will investigate:
does the model still perform this well on longer, more
creative human-written jailbreaks where the harmful content
is buried deeper in the text?